In [1]:
import zarr
import re
import os
import gc
import time

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import dask.array as da

from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar
from sklearn.model_selection import train_test_split

In [2]:
def sanitize_key(key):
    return re.sub(r'[^\w]', '_', str(key)).strip('_')

**Preprocess data and select HVGs**

In [30]:
rng = np.random.default_rng(seed=42)

adata = sc.read_h5ad('/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/lucav2/core_250130.h5ad')
adata.X = adata.layers['count']
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
idx = np.sort(rng.choice(np.arange(adata.shape[1]), size=5983))
adata_hvg = adata[:, idx].copy()

/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


**Prepare combinations**

In [34]:
adata_hvg.obs['cell_id'] = adata_hvg.obs.index.astype(str)
adata_hvg.obs.reset_index(drop=True, inplace=True)
for col in ['cell_type','dataset','origin']:
    adata_hvg.obs[col] = adata_hvg.obs[col].astype(str)

adata_hvg.obs['comb_key'] = adata_hvg.obs.apply(
    lambda r: "-".join([
        sanitize_key(r['cell_type']),
        sanitize_key(r['dataset']),
        sanitize_key(r['origin'])
    ]),
    axis=1
)

In [35]:
output_dir = '/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_random_hvgs/comb_w_obs.zarr'
os.makedirs(output_dir, exist_ok=True)
n_vars = adata_hvg.X.shape[1]

root_zarr = zarr.open(output_dir, mode='a')
root_zarr.attrs['var_names'] = adata_hvg.var['feature_name'].tolist()

current_combinations = set()

for combination_key, group in tqdm(adata_hvg.obs.groupby('comb_key')):
    current_combinations.add(combination_key)
    x_group = adata_hvg.X[group.index].toarray()

    da.from_array(x_group, chunks=-1).astype(adata_hvg.X.dtype).to_zarr(output_dir+'/'+combination_key, component='X', overwrite=True)
    da.from_array(group['cell_id'].values, chunks=-1).astype('<U40').to_zarr(output_dir+'/'+combination_key, component='obs_index', overwrite=True)

existing_combinations = set(root_zarr.attrs.get('combinations', []))
existing_combinations.update(current_combinations)
root_zarr.attrs['combinations'] = sorted(existing_combinations)

OSError: [Errno 122] Disk quota exceeded

**Preparing split01**

In [ ]:
CHUNK_SIZE = 50_000

source_dir = '/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_random_hvgs/comb_w_obs.zarr'
output_dir = "/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_random_hvgs/split01"

train_ratio=0.7
val_ratio=0.15
test_ratio=0.15

seed=42

meta_data = zarr.open('/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/lucav2/split01/split_metadata.zarr')
train_combs = meta_data.attrs['train_combinations']
val_combs = meta_data.attrs['val_combinations']
test_combs = meta_data.attrs['test_combinations']

meta_save_path = os.path.join(output_dir, "split_metadata.zarr")
meta_group = zarr.open_group(meta_save_path, mode='w')
meta_group.attrs['train_combinations'] = train_combs
meta_group.attrs['val_combinations'] = val_combs
meta_group.attrs['test_combinations'] = test_combs
meta_group.attrs['random_seed'] = seed

print("Processing validation data...")
val_arrays = []
for comb in val_combs:
    comb_path = os.path.join(comb_dir, comb)
    val_arrays.append(da.from_zarr(comb_path, 'X'))

val_data = da.concatenate(val_arrays, axis=0)
print(f"Validation data shape: {val_data.shape}")

val_save_path = os.path.join(output_dir, 'val.zarr')
print(f"Saving validation data to {val_save_path}")
with ProgressBar():
    val_data.rechunk((CHUNK_SIZE, val_data.shape[1])).to_zarr(val_save_path, component='X', overwrite=True)
print(f"Done saving validation data")

del val_arrays, val_data
gc.collect()
time.sleep(10)

print("Processing test data...")
test_arrays = []
for comb in test_combs:
    comb_path = os.path.join(comb_dir, comb)
    test_arrays.append(da.from_zarr(comb_path, 'X'))

test_data = da.concatenate(test_arrays,  axis=0)
print(f"Test data shape: {test_data.shape}")

test_save_path = os.path.join(output_dir, 'test.zarr')
print(f"Saving test data to {test_save_path}")
with ProgressBar():
    test_data.rechunk((CHUNK_SIZE, test_data.shape[1])).to_zarr(test_save_path, component='X', overwrite=True)
print(f"Done saving test data")

del test_arrays, test_data
gc.collect()
time.sleep(10)

print("Processing training data...")
train_arrays = []
for comb in train_combs:
    comb_path = os.path.join(comb_dir, comb)
    train_arrays.append(da.from_zarr(comb_path, 'X'))

train_data = da.concatenate(train_arrays, axis=0)
print(f"Training data shape: {train_data.shape}")

n_vars = train_data.shape[1]
train_data = train_data.rechunk((CHUNK_SIZE, n_vars))
print(f"Training data rechunked: {train_data.numblocks} blocks")

train_unsfd_save_path = os.path.join(output_dir, 'train_unsfd.zarr')
print(f"Saving unshuffled training data to {train_unsfd_save_path}")
with ProgressBar():
    train_data.to_zarr(train_unsfd_save_path, component='X', overwrite=True)
print(f"Done saving unshuffled training data")

print("Shuffling training data...")
rng = np.random.default_rng(seed=seed)
n = train_data.shape[0]
perm = rng.permutation(n)
indexer = [list(perm[i:i+CHUNK_SIZE]) for i in range(0, n, CHUNK_SIZE)]

train_data_shuffled = da.shuffle(train_data, indexer=indexer, axis=0, chunks='auto')
print(f"Training data shuffled: {train_data_shuffled.numblocks} blocks")

train_save_path = os.path.join(output_dir, 'train.zarr')
with ProgressBar():
    train_data_shuffled.to_zarr(train_save_path, component='X', overwrite=True)
print(f"Done saving shuffled training data")

del train_arrays, train_data, train_data_shuffled
gc.collect()
time.sleep(10)

**Preparing split02**

In [ ]:
CHUNK_SIZE = 50_000

source_dir = '/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_random_hvgs/comb_w_obs.zarr'
output_dir = "/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_random_hvgs/split02"

train_ratio=0.7
val_ratio=0.15
test_ratio=0.15

seed=42

meta_data = zarr.open('/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/lucav2/split02/split_metadata.zarr')
train_combs = meta_data.attrs['train_combinations']
val_combs = meta_data.attrs['val_combinations']
test_combs = meta_data.attrs['test_combinations']

meta_save_path = os.path.join(output_dir, "split_metadata.zarr")
meta_group = zarr.open_group(meta_save_path, mode='w')
meta_group.attrs['train_combinations'] = train_combs
meta_group.attrs['val_combinations'] = val_combs
meta_group.attrs['test_combinations'] = test_combs
meta_group.attrs['random_seed'] = seed

print("Processing validation data...")
val_arrays = []
for comb in val_combs:
    comb_path = os.path.join(comb_dir, comb)
    val_arrays.append(da.from_zarr(comb_path, 'X'))

val_data = da.concatenate(val_arrays, axis=0)
print(f"Validation data shape: {val_data.shape}")

val_save_path = os.path.join(output_dir, 'val.zarr')
print(f"Saving validation data to {val_save_path}")
with ProgressBar():
    val_data.rechunk((CHUNK_SIZE, val_data.shape[1])).to_zarr(val_save_path, component='X', overwrite=True)
print(f"Done saving validation data")

del val_arrays, val_data
gc.collect()
time.sleep(10)

print("Processing test data...")
test_arrays = []
for comb in test_combs:
    comb_path = os.path.join(comb_dir, comb)
    test_arrays.append(da.from_zarr(comb_path, 'X'))

test_data = da.concatenate(test_arrays,  axis=0)
print(f"Test data shape: {test_data.shape}")

test_save_path = os.path.join(output_dir, 'test.zarr')
print(f"Saving test data to {test_save_path}")
with ProgressBar():
    test_data.rechunk((CHUNK_SIZE, test_data.shape[1])).to_zarr(test_save_path, component='X', overwrite=True)
print(f"Done saving test data")

del test_arrays, test_data
gc.collect()
time.sleep(10)

print("Processing training data...")
train_arrays = []
for comb in train_combs:
    comb_path = os.path.join(comb_dir, comb)
    train_arrays.append(da.from_zarr(comb_path, 'X'))

train_data = da.concatenate(train_arrays, axis=0)
print(f"Training data shape: {train_data.shape}")

n_vars = train_data.shape[1]
train_data = train_data.rechunk((CHUNK_SIZE, n_vars))
print(f"Training data rechunked: {train_data.numblocks} blocks")

train_unsfd_save_path = os.path.join(output_dir, 'train_unsfd.zarr')
print(f"Saving unshuffled training data to {train_unsfd_save_path}")
with ProgressBar():
    train_data.to_zarr(train_unsfd_save_path, component='X', overwrite=True)
print(f"Done saving unshuffled training data")

print("Shuffling training data...")
rng = np.random.default_rng(seed=seed)
n = train_data.shape[0]
perm = rng.permutation(n)
indexer = [list(perm[i:i+CHUNK_SIZE]) for i in range(0, n, CHUNK_SIZE)]

train_data_shuffled = da.shuffle(train_data, indexer=indexer, axis=0, chunks='auto')
print(f"Training data shuffled: {train_data_shuffled.numblocks} blocks")

train_save_path = os.path.join(output_dir, 'train.zarr')
with ProgressBar():
    train_data_shuffled.to_zarr(train_save_path, component='X', overwrite=True)
print(f"Done saving shuffled training data")

del train_arrays, train_data, train_data_shuffled
gc.collect()
time.sleep(10)

**Preparing split03**

In [ ]:
CHUNK_SIZE = 50_000

source_dir = '/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_random_hvgs/comb_w_obs.zarr'
output_dir = "/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_random_hvgs/split03"

train_ratio=0.7
val_ratio=0.15
test_ratio=0.15

seed=42

meta_data = zarr.open('/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_random_hvgs/split03/split_metadata.zarr')
train_combs = meta_data.attrs['train_combinations']
val_combs = meta_data.attrs['val_combinations']
test_combs = meta_data.attrs['test_combinations']

meta_save_path = os.path.join(output_dir, "split_metadata.zarr")
meta_group = zarr.open_group(meta_save_path, mode='w')
meta_group.attrs['train_combinations'] = train_combs
meta_group.attrs['val_combinations'] = val_combs
meta_group.attrs['test_combinations'] = test_combs
meta_group.attrs['random_seed'] = seed

print("Processing validation data...")
val_arrays = []
for comb in val_combs:
    comb_path = os.path.join(comb_dir, comb)
    val_arrays.append(da.from_zarr(comb_path, 'X'))

val_data = da.concatenate(val_arrays, axis=0)
print(f"Validation data shape: {val_data.shape}")

val_save_path = os.path.join(output_dir, 'val.zarr')
print(f"Saving validation data to {val_save_path}")
with ProgressBar():
    val_data.rechunk((CHUNK_SIZE, val_data.shape[1])).to_zarr(val_save_path, component='X', overwrite=True)
print(f"Done saving validation data")

del val_arrays, val_data
gc.collect()
time.sleep(10)

print("Processing test data...")
test_arrays = []
for comb in test_combs:
    comb_path = os.path.join(comb_dir, comb)
    test_arrays.append(da.from_zarr(comb_path, 'X'))

test_data = da.concatenate(test_arrays,  axis=0)
print(f"Test data shape: {test_data.shape}")

test_save_path = os.path.join(output_dir, 'test.zarr')
print(f"Saving test data to {test_save_path}")
with ProgressBar():
    test_data.rechunk((CHUNK_SIZE, test_data.shape[1])).to_zarr(test_save_path, component='X', overwrite=True)
print(f"Done saving test data")

del test_arrays, test_data
gc.collect()
time.sleep(10)

print("Processing training data...")
train_arrays = []
for comb in train_combs:
    comb_path = os.path.join(comb_dir, comb)
    train_arrays.append(da.from_zarr(comb_path, 'X'))

train_data = da.concatenate(train_arrays, axis=0)
print(f"Training data shape: {train_data.shape}")

n_vars = train_data.shape[1]
train_data = train_data.rechunk((CHUNK_SIZE, n_vars))
print(f"Training data rechunked: {train_data.numblocks} blocks")

train_unsfd_save_path = os.path.join(output_dir, 'train_unsfd.zarr')
print(f"Saving unshuffled training data to {train_unsfd_save_path}")
with ProgressBar():
    train_data.to_zarr(train_unsfd_save_path, component='X', overwrite=True)
print(f"Done saving unshuffled training data")

print("Shuffling training data...")
rng = np.random.default_rng(seed=seed)
n = train_data.shape[0]
perm = rng.permutation(n)
indexer = [list(perm[i:i+CHUNK_SIZE]) for i in range(0, n, CHUNK_SIZE)]

train_data_shuffled = da.shuffle(train_data, indexer=indexer, axis=0, chunks='auto')
print(f"Training data shuffled: {train_data_shuffled.numblocks} blocks")

train_save_path = os.path.join(output_dir, 'train.zarr')
with ProgressBar():
    train_data_shuffled.to_zarr(train_save_path, component='X', overwrite=True)
print(f"Done saving shuffled training data")

del train_arrays, train_data, train_data_shuffled
gc.collect()
time.sleep(10)